# 1. Configuração do Ambiente e Base de Dados

Executando o bloco abaixo no ambiente para importar as bibliotecas e gerar o DataFrame com as coordenadas geográficas simuladas:

In [1]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster

# Gerando dados sintéticos de imóveis em Nova Iguaçu e Queimados
np.random.seed(42)
n_imoveis = 45

# Coordenadas base (Aproximadas)
# Nova Iguaçu: -22.756, -43.460
# Queimados: -22.716, -43.555

dados_imoveis = {
    'id_imovel': range(1, n_imoveis + 1),
    'cidade': np.where(np.random.rand(n_imoveis) > 0.4, 'Nova Iguaçu', 'Queimados'),
    'valor_venda': np.random.uniform(150000, 850000, n_imoveis).round(2),
    'tipo': np.random.choice(['Casa', 'Apartamento', 'Terreno'], n_imoveis)
}

df_mapa = pd.DataFrame(dados_imoveis)

# Atribuindo coordenadas com base na cidade adicionando uma pequena dispersão aleatória
def gerar_lat(cidade):
    if cidade == 'Nova Iguaçu':
        return -22.756 + np.random.uniform(-0.03, 0.03)
    return -22.716 + np.random.uniform(-0.02, 0.02)

def gerar_lon(cidade):
    if cidade == 'Nova Iguaçu':
        return -43.460 + np.random.uniform(-0.03, 0.03)
    return -43.555 + np.random.uniform(-0.02, 0.02)

df_mapa['latitude'] = df_mapa['cidade'].apply(gerar_lat)
df_mapa['longitude'] = df_mapa['cidade'].apply(gerar_lon)


# 2. Roteiro de Tarefas

**Parte 1: Inicialização e Marcadores Básicos**


In [ ]:
lat_media = df_mapa['latitude'].mean()
lon_media = df_mapa['longitude'].mean()

mapa = folium.Map(
    location=[lat_media, lon_media],
    zoom_start=12,
    tiles='OpenStreetMap'
)

for _, imovel in df_mapa.head(5).iterrows():
    popup_texto = (
        f"Tipo: {imovel['tipo']}<br>"
        f"Valor de venda: R$ {imovel['valor_venda']:,.2f}"
    )

    folium.Marker(
        location=[imovel['latitude'], imovel['longitude']],
        popup=folium.Popup(popup_texto, max_width=300)
    ).add_to(mapa)

mapa


**Parte 2: Customização Visual com Marcadores Circulares**

In [ ]:
lat_media = df_mapa['latitude'].mean()
lon_media = df_mapa['longitude'].mean()

mapa_circulos = folium.Map(
    location=[lat_media, lon_media],
    zoom_start=12,
    tiles='OpenStreetMap'
)

for _, imovel in df_mapa.iterrows():

    cor = 'blue' if imovel['cidade'] == 'Nova Iguaçu' else 'orange'

    folium.CircleMarker(
        location=[imovel['latitude'], imovel['longitude']],
        radius=8,
        color=cor,
        fill=True,
        fill_color=cor,
        fill_opacity=0.7,
        tooltip='Clique para detalhes'
    ).add_to(mapa_circulos)

mapa_circulos

**Parte 3: Agrupamento Inteligente (Clustering)**

In [ ]:
lat_media = df_mapa['latitude'].mean()
lon_media = df_mapa['longitude'].mean()

mapa_cluster = folium.Map(
    location=[lat_media, lon_media],
    zoom_start=12,
    tiles='OpenStreetMap'
)

cluster = MarkerCluster().add_to(mapa_cluster)

cores_tipo = {
    'Casa': 'green',
    'Apartamento': 'blue',
    'Terreno': 'gray'
}

for _, imovel in df_mapa.iterrows():

    cor = cores_tipo[imovel['tipo']]

    popup_texto = (
        f"Tipo: {imovel['tipo']}<br>"
        f"Valor de venda: R$ {imovel['valor_venda']:,.2f}<br>"
        f"Cidade: {imovel['cidade']}"
    )

    marcador = folium.Marker(
        location=[imovel['latitude'], imovel['longitude']],
        popup=folium.Popup(popup_texto, max_width=300),
        icon=folium.Icon(
            color=cor,
            icon='home',
            prefix='glyphicon'
        )
    )

    marcador.add_to(cluster)

mapa_cluster.save('mapa_imoveis_baixada.html')

mapa_cluster